#**Tuteur IA**

## **1.Conception**

###Présentation générale du Notebook

Ce notebook présente l’**implémentation du Tuteur IA**, utilisé pour assister le développement de l’IA apprenante.  
Il repose sur une architecture de type **RAG (Retrieval-Augmented Generation)**, combinant une base documentaire, une recherche sémantique et un modèle de langage (LLM).

---

## *Pré-requis*

Avant d’exécuter le notebook :

- Importer le dossier **`Base_documentaire`** dans la section **Fichiers** de Google Colab.
- Exécuter **toutes les cellules** du notebook pour démarrer le Tuteur IA.
- L’interaction avec le tuteur se fait dans la dernière section : **`3. Main`**.

---

## *Organisation du Notebook*

Le notebook est structuré en trois sections principales, dont la section centrale d’implémentation est divisée en plusieurs parties.

---

## *Installation et configuration*

### A. Installation des bibliothèques
Cette partie installe et importe l’ensemble des bibliothèques nécessaires au fonctionnement du système (traitement de documents, embeddings, base vectorielle, reranker et LLM).

### B. Configuration globale
Cette section définit :
- Les modèles utilisés (embeddings, LLM, reranker)
- Les paramètres globaux :
  - taille des chunks
  - nombre de documents récupérés
  - paramètres de génération du modèle

---

## *Implémentation du pipeline RAG*

### C. Extraction et traitement des documents
Cette partie gère la **récupération de l’information** à partir de la base documentaire :

- Lecture des fichiers PDF depuis le dossier **`BaseDocumentaire`**
- Découpage des documents en chunks
- Génération des embeddings
- Stockage des vecteurs dans la base vectorielle **ChromaDB**
- Recherche des chunks pertinents à partir de la question utilisateur

Afin d’obtenir un **contexte plus pertinent**, il est possible :
- d’enrichir la question utilisateur (la rendre plus explicative)
- d’appliquer un **reranker** sur les chunks récupérés

---

### D. Intégration du modèle de langage (LLM)
Cette section :
- Initialise le modèle de langage
- Définit le prompt d’instruction à suivre
- Appelle le LLM en lui fournissant le **contexte enrichi**

---

## 🌱 *Mesure de l’empreinte carbone*

La mesure carbone a été réalisée uniquement sur les **fonctions les plus critiques**, afin d’évaluer leur impact énergétique de manière pertinente.

### Fonctions mesurées avec CodeCarbon
- Génération des embeddings
- Recherche dans la base vectorielle
- Application du reranker
- Appel au modèle de langage avec contexte

### Fonctions non mesurées
- Import et installation des bibliothèques
- Chargement des fichiers
- Configuration statique
- Affichage et interaction utilisateur

Ces parties ont un coût énergétique négligeable et n’influencent pas significativement l’empreinte carbone globale.


## **2.Implémentation**

###A-Installation des bibliothèques


In [ ]:
!pip -q install \
    chromadb \
    sentence-transformers \
    transformers \
    accelerate \
    pypdf2 \
    codecarbon \
    langchain-text-splitters

###B-Configuration globale

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
import PyPDF2
from typing import Optional, List, Dict, Tuple
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from codecarbon import OfflineEmissionsTracker

# ============================================================================
# CONFIGURATION
# ============================================================================

EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.1"


CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
SIMILARITY_THRESHOLD = 0.5
COLLECTION_NAME = "rag_lab_collection"

# ============================================================================
# VARIABLES GLOBALES
# ============================================================================

model = None
client = None
collection = None
text_splitter = None
reranker = None
tokenizer = None
llm = None

###C-Extraction / récupération de documents pertinents

####C.1-Initialisation du rag_partie_récupération

In [ ]:
# ============================================================================
# INITIALISATION
# ============================================================================

def initialiser_rag(similarity_threshold=0.5):
    """
    Initialise le système RAG avec toutes ses dépendances.

    Args:
        similarity_threshold: Seuil de similarité pour filtrer les résultats
    """
    global model, client, collection, text_splitter, SIMILARITY_THRESHOLD, reranker

    print("Initialisation du système RAG...")

    # Modèle d'embedding
    model = SentenceTransformer(EMBEDDING_MODEL)

    # Reranker
    reranker = CrossEncoder(RERANKER_MODEL)

    # Base de données ChromaDB
    client = chromadb.Client()
    collection = client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"}
    )

    # Text splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", " "]
    )

    # Seuil de similarité
    SIMILARITY_THRESHOLD = similarity_threshold

    print("Système RAG initialisé avec succès")

####C.2-Lecture des documents

In [ ]:
import re

# ============================================================================
# Reduire bruit dans texte
# ============================================================================
def nettoyer_texte_pdf(text: str) -> str:
    """
    Nettoie le texte extrait d'un PDF pour réduire le bruit
    (sommaire, mots cassés, espaces, artefacts d'accents).
    À appeler AVANT le chunking.
    """

    # 1) Supprimer le sommaire (ici spécifique à ton doc "TRANSFORMERS SOMMAIRE ...")
    #    On coupe tout ce qui est avant le premier vrai titre de section, ex: "1- M OTIVATION" ou "1- MOTIVATION"
    sommaire_match = re.search(r"1-\s*M ?OTIVATION", text, flags=re.IGNORECASE)
    if sommaire_match:
        text = text[sommaire_match.start():]

    # 2) Normaliser les retours à la ligne (si tu les as encore à cette étape)
    #    Si tu as déjà fait ça ailleurs, tu peux sauter cette partie.
    text = text.replace('\r', ' ')
    text = text.replace('\n', ' ')

    # 3) Supprimer les numéros de page collés au texte (ex : "1possible", "4repr´esentations")
    #    On enlève un chiffre suivi directement d'une lettre, quand ça ressemble à un "début de page"
    text = re.sub(r"\b\d+(?=[A-Za-z´`])", " ", text)

    # 4) Corriger quelques artefacts d'accents fréquents dans ton corpus
    corrections = {
        "M´ecanisme": "Mécanisme",
        "repr ésentation": "représentation",
        "repr ´esentation": "représentation",
        "r éseau": "réseau",
        "r éseaux": "réseaux",
        "donn ées": "données",
        "adapt és": "adaptés",
        "pr éc´edemment": "précédemment",
        "t ˆaches": "tâches",
        "entr ée": "entrée",
        "entr ´ee": "entrée",
        "langues na- turelles": "langues naturelles",
    }
    for src, tgt in corrections.items():
        text = text.replace(src, tgt)

    # 5) Corriger des motifs plus génériques : " ´e" -> "é", " ´a" -> "à", etc.
    #    Ça reste un peu bourrin, mais généralement ça améliore bien.
    text = text.replace(" ´e", "é")
    text = text.replace(" ´a", "à")
    text = text.replace(" ´i", "í")  # peu fréquent en français, à adapter
    text = text.replace(" ´o", "ó")
    text = text.replace(" ´u", "ú")

    # Si tu vois souvent "`a" pour "à", tu peux aussi faire :
    text = text.replace(" `a", " à")
    text = text.replace(" `e", " è")

    # 6) Normaliser les espaces (un seul espace partout)
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [ ]:
# ============================================================================
# CHARGEMENT DE DOCUMENTS
# ============================================================================

def charger_pdf(file_path: str) -> str:
    """
    Charge et extrait le contenu d'un fichier PDF.

    Args:
        file_path: Chemin vers le fichier PDF

    Returns:
        Contenu textuel du PDF
    """
    with open(file_path, 'rb') as f:
        pdf_reader = PyPDF2.PdfReader(f)
        content = ""
        for page in pdf_reader.pages:
            content += page.extract_text()
    return content


def charger_texte(file_path: str) -> str:
    """
    Charge le contenu d'un fichier texte.

    Args:
        file_path: Chemin vers le fichier texte

    Returns:
        Contenu du fichier
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        return f.read()


def charger_fichier(file_path: str):
    """
    Charge un fichier (PDF ou TXT) et l'ajoute à la base vectorielle.

    Args:
        file_path: Chemin vers le fichier à charger

    Raises:
        FileNotFoundError: Si le fichier n'existe pas
        ValueError: Si le type de fichier n'est pas supporté
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Le fichier '{file_path}' n'existe pas.")

    file_name = os.path.basename(file_path)
    file_extension = os.path.splitext(file_path)[1].lower()

    # Lecture selon le type
    if file_extension == '.pdf':
        content = nettoyer_texte_pdf(charger_pdf(file_path))
        file_type = 'pdf'
    elif file_extension == '.txt':
        content = charger_texte(file_path)
        file_type = 'texte'
    else:
        raise ValueError(f"Type de fichier non supporté: {file_extension}")

    if not content.strip():
        print("Attention: Le fichier est vide.")
        return

    # Créer le document
    document = {
        "content": content,
        "titre": file_name,
        "type": file_type
    }

    ajouter_documents([document])


def charger_dossier(chemin_dossier: str, extensions: List[str] = ['.pdf', '.txt']) -> int:
    """
    Charge tous les fichiers d'un dossier dans la base vectorielle.

    Args:
        chemin_dossier: Chemin vers le dossier contenant les documents
        extensions: Liste des extensions de fichiers à charger

    Returns:
        Nombre de fichiers chargés avec succès

    Raises:
        FileNotFoundError: Si le dossier n'existe pas
        ValueError: Si le chemin n'est pas un dossier
    """
    if not os.path.exists(chemin_dossier):
        raise FileNotFoundError(f"Le dossier '{chemin_dossier}' n'existe pas.")

    if not os.path.isdir(chemin_dossier):
        raise ValueError(f"'{chemin_dossier}' n'est pas un dossier.")

    # Lister tous les fichiers du dossier
    fichiers = os.listdir(chemin_dossier)

    # Filtrer selon les extensions autorisées
    fichiers_valides = [
        f for f in fichiers
        if os.path.splitext(f)[1].lower() in extensions
    ]

    if not fichiers_valides:
        print(f"Aucun fichier {extensions} trouvé dans '{chemin_dossier}'")
        return 0

    print(f"{len(fichiers_valides)} fichier(s) trouvé(s) dans '{chemin_dossier}'")

    # Charger chaque fichier
    nb_charges = 0
    for fichier in fichiers_valides:
        chemin_complet = os.path.join(chemin_dossier, fichier)
        try:
            print(f"  Chargement de {fichier}...")
            charger_fichier(chemin_complet)
            nb_charges += 1
        except Exception as e:
            print(f"  Erreur lors du chargement de {fichier}: {e}")

    print(f"{nb_charges}/{len(fichiers_valides)} fichier(s) chargé(s) avec succès\n")
    return nb_charges

####C.3-Traitement des documents

In [ ]:
# ============================================================================
# TRAITEMENT DES DOCUMENTS
# ============================================================================

def decouper_texte(texte: str) -> List[str]:
    """
    Découpe un texte en chunks.

    Args:
        texte: Texte à découper

    Returns:
        Liste de chunks de texte
    """
    return text_splitter.split_text(texte)


def ajouter_documents(raw_documents: List[Dict]):
    """
    Découpe les documents et les stocke dans la base vectorielle.

    Args:
        raw_documents: Liste de dictionnaires contenant les documents
    """
    tracker_train = OfflineEmissionsTracker(country_iso_code="FRA", project_name=f"{EMBEDDING_MODEL} : Chunking and Embedding and storing in ChromaDB")
    tracker_train.start()

    ids, contents, metadatas = [], [], []

    for doc in raw_documents:
        chunks = decouper_texte(doc["content"])
        doc_id = doc.get("titre", "unknown")

        for i, chunk in enumerate(chunks):
            ids.append(f"{doc_id}_chunk_{i}")
            contents.append(chunk)
            metadatas.append({
                "title": doc.get("titre", "Sans titre"),
                "type": doc.get("type", "inconnu")
            })

    # Générer les embeddings
    embeddings = model.encode(contents).tolist()



    # Stocker dans ChromaDB
    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=contents,
        metadatas=metadatas
    )

    tracker_train.stop()

    print(f"{len(ids)} segments stockés avec succès")

# ============================================================================
# GESTION DE LA BASE DE DONNÉES
# ============================================================================

def vider_base():
    """Vide complètement la base de données."""
    global collection

    client.delete_collection(COLLECTION_NAME)
    collection = client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"}
    )
    print("Base de données vidée")


def compter_documents() -> int:
    """
    Retourne le nombre de chunks dans la base.

    Returns:
        Nombre de documents dans la collection
    """
    return collection.count()

####C.4-Enrichissement de la question

In [ ]:
# ============================================================================
# RECHERCHE ET RERANKING
# ============================================================================

def enrichir_requete(query: str) -> str:
    """
    Enrichit une requête avec des termes génériques.

    Args:
        query: Requête originale

    Returns:
        Requête enrichie
    """
    return query + " explication détaillée, définition du principe et  fonctionnement mécanisme"


def calculer_similarite_cosinus(embeddings1: np.ndarray, embeddings2: np.ndarray) -> np.ndarray:
    """
    Calcule la similarité cosinus entre deux ensembles d'embeddings.

    Args:
        embeddings1: Premier ensemble d'embeddings
        embeddings2: Second ensemble d'embeddings

    Returns:
        Scores de similarité cosinus
    """
    embeddings1 = np.array(embeddings1)
    embeddings2 = np.array(embeddings2)

    dot = (embeddings1 @ embeddings2)
    norms = np.linalg.norm(embeddings1) * np.linalg.norm(embeddings2)

    return dot / (norms + 1e-8)

####C.5-Reranker

In [ ]:
def reranker_chunks(query: str, chunks_avec_scores: List[Tuple[str, float]]) -> List[Tuple[str, float]]:
    """
    Réordonne les chunks avec un Cross-Encoder pour plus de précision,
    sans modifier les scores initiaux (on garde les scores d'embeddings
    pour la confiance).
    """
    if not chunks_avec_scores:
        return []

    textes = [chunk for chunk, _ in chunks_avec_scores]
    paires = [(query, texte) for texte in textes]

    scores_rerank = reranker.predict(paires)

    indices_tries = sorted(
        range(len(textes)),
        key=lambda i: scores_rerank[i],
        reverse=True
    )

    return [chunks_avec_scores[i] for i in indices_tries]

####C.6-Obtenir Meilleur Contextes

In [ ]:
def detect_question_type(query: str) -> str:
    q = query.lower()

    definition_keywords = [
        "c quoi", "c'est quoi", "qu'est-ce que",
        "que signifie", "définition"
    ]

    explanation_keywords = [
        "comment", "pourquoi", "fonctionne", "fonctionnement",
        "mécanisme", "principe", "explication", "explique"
    ]

    comparison_keywords = [
        "différence", "différence entre", "comparaison",
        "vs", "versus", "par rapport à"
    ]

    if any(k in q for k in definition_keywords):
        return "definition"
    elif any(k in q for k in comparison_keywords):
        return "comparaison"
    elif any(k in q for k in explanation_keywords):
        return "explication"
    else:
        return "autre"

def obtenir_contexte_pertinent(
    query: str,
    top_k: int = 3,
) -> Tuple[Optional[str], Optional[str]]:
    """
    Récupère le contexte le plus pertinent pour une requête.
    Retourne un tuple (contexte, confiance) :
        - En cas de succès  : (texte_contexte, "Confiance: XX%")
        - En cas d'échec    : (None, message_erreur)
    """

    tracker_train = OfflineEmissionsTracker(country_iso_code="FRA", project_name=f"Obtenir meilleur Contexte avec {EMBEDDING_MODEL} et {RERANKER_MODEL}")
    tracker_train.start()

    # --- Vérification préalable ---
    if collection.count() == 0:
        return None, "La base de données est vide. Ajoutez d'abord des documents."

    # --- Encoding de la requête brute ---
    query_vector = model.encode([query])[0]

    # --- Rappel large depuis ChromaDB ---
    candidate_k = max(top_k * 4, 10)
    results = collection.query(
        query_embeddings=query_vector,
        n_results=candidate_k,
    )

    if not results["distances"][0]:
        return None, "Aucun résultat trouvé."

    docs = results["documents"][0]
    distances = results["distances"][0]

    # --- Calcul des scores de similarité ---
    question_type = detect_question_type(query)

    if question_type == "autre":
        # Chemin enrichi : on re-encode avec une requête enrichie
        enriched_query = enrichir_requete(query)
        enriched_embedding = model.encode([enriched_query])[0]
        chunk_embeddings = model.encode(docs)
        sim_scores = calculer_similarite_cosinus(chunk_embeddings, enriched_embedding)
    else:
        # Chemin simple : conversion directe des distances ChromaDB
        sim_scores = [1 - d for d in distances]

    # --- Tri par score décroissant ---
    ranked = sorted(
        zip(docs, sim_scores),
        key=lambda x: x[1],
        reverse=True,
    )

    # --- Filtrage par seuil + déduplication ---
    chunks_avec_scores: List[Tuple[str, float]] = []
    seen_content: set = set()

    for doc, score in ranked:
        doc_hash = hash(doc[:100])

        if doc_hash in seen_content:
            continue
        seen_content.add(doc_hash)

        if score >= SIMILARITY_THRESHOLD:
            chunks_avec_scores.append((doc, score))

    if not chunks_avec_scores:
        return "No contexte", "No confiance"

    # --- Reranking et sélection finale ---
    chunks_avec_scores = reranker_chunks(query, chunks_avec_scores)
    chunks_avec_scores = chunks_avec_scores[:top_k]

    tracker_train.stop()

    # --- Construction du contexte de sortie ---
    relevant_docs = [chunk for chunk, _ in chunks_avec_scores]
    relevant_scores = [score for _, score in chunks_avec_scores]

    context = "\n----------Next chunk---------\n".join(relevant_docs)
    confidence = f"Confiance: {max(relevant_scores):.2%}"

    return context, confidence

### D-Intégration du **LLM**

####D.1-Initialisation du LLM

In [ ]:
def initialiser_llm():
    """Initialise le modèle de langage pour la génération de réponses."""
    global tokenizer, llm

    print("Chargement du modèle de langage...")

    # Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)

    # Modèle (float16 pour économiser la VRAM)
    model_llm = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float16
    )

    # Pipeline de génération
    llm = pipeline(
        task="text-generation",
        model=model_llm,
        tokenizer=tokenizer,
        temperature=0.2,
        top_p=0.9,
        max_new_tokens=1024,
    )

    print("Modèle de langage chargé avec succès")

####D.2-Configuration du prompt du LLM

In [ ]:
# ============================================================================
# GÉNÉRATION DE RÉPONSES
# ============================================================================

def appel_modele(question: str, context: str) -> str:
    """
    Génère une réponse en utilisant le contexte fourni ou les connaissances générales.

    Args:
        question: Question de l'utilisateur
        context: Contexte récupéré par RAG (ou "No contexte")

    Returns:
        Réponse générée par le modèle
    """
    using_contexte = context != "No contexte"

    if using_contexte:
        prompt = f"""### Instruction:
Tu es un tuteur IA spécialisé en Python, scikit-learn, TensorFlow et PyTorch.

Utilise le contexte fourni pour répondre à la question. Si le contexte n'est pas pertinent, utilise tes connaissances.

Contexte:
{context}

Question: {question}

### Response:
"""
    else:
        prompt = f"""### Instruction:
Tu es un tuteur IA spécialisé en Python, scikit-learn, TensorFlow et PyTorch.
Si la question n'est pas parmi tes spécialités (python, IA etc), tu es obligé de dire 'je ne suis pas capable de répondre à cette question car elle n'est pas dans ma spécialité'.
Réponds à la question en te basant sur tes connaissances générales.

Question: {question}

### Response:
"""
    tracker_train = OfflineEmissionsTracker(country_iso_code="FRA", project_name=f"{LLM_MODEL_NAME} : Génération de réponse avec ou sans contexte")
    tracker_train.start()

    # Génération
    output = llm(
        prompt,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        num_beams=1,
        num_return_sequences=1,
        return_full_text=False,
        max_new_tokens=1024,
    )[0]["generated_text"]

    tracker_train.stop()
    # Nettoyage de la réponse
    print(len(output.split()))
    output = output.strip()

    # Ajout du préfixe selon la source
    if using_contexte:
        prefix = "Basé sur le contexte fourni:\n\n"
    else:
        prefix = "Basé sur mes connaissances générales:\n\n"

    return prefix + output



## 3.Main

In [ ]:
# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":

    print("Initialisation du système RAG...")
    initialiser_rag(similarity_threshold=0.5)

    print("Chargement du modèle de langage...")
    initialiser_llm()

    print("Chargement des documents...")
    charger_dossier("/content/BaseDocumentaire")

    print("\n" + "=" * 60)
    print("Tuteur IA - Python & Machine Learning")
    print("=" * 60)
    print("Tapez 'exit' ou 'quit' pour quitter")
    print("Tapez 'clear' pour vider l'historique")
    print("=" * 60 + "\n")

    while True:

        user_query_RAG = input("Votre question: ").strip()

        if user_query_RAG.lower() in ['exit', 'quit', 'q']:
            print("\nÀ bientôt!")
            break

        if user_query_RAG.lower() == 'clear':
            vider_base()
            print("Base de données vidée.\n")
            continue

        if not user_query_RAG:
            continue

        print("\nRecherche du contexte...")

        context_RAG, confidence = obtenir_contexte_pertinent(
            user_query_RAG,
            top_k=3,
        )

        # Affichage du contexte (optionnel)
        print("\n=== Contexte RAG ===")
        print(context_RAG)
        if confidence:
            print(confidence)

        # Génération de la réponse
        print("\nGénération de la réponse...\n")

        resp_RAG = appel_modele(user_query_RAG, context_RAG)

        print("\n" + "=" * 60)
        print("RÉPONSE")
        print("=" * 60)
        print(resp_RAG)
        print("\n" + "=" * 60 + "\n")

Initialisation du système RAG...
Initialisation du système RAG...
Système RAG initialisé avec succès
Chargement du modèle de langage...
Chargement du modèle de langage...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Device set to use cuda:0


Modèle de langage chargé avec succès
Chargement des documents...
11 fichier(s) trouvé(s) dans '/content/BaseDocumentaire'
  Chargement de [Cours] _Réseaux Convolutifs.pdf...
94 segments stockés avec succès
  Chargement de [Cours]_RÉSEAUX ADVERSAIRES GÉNÉRATIFS.pdf...
59 segments stockés avec succès
  Chargement de [Cours]_Autoencoderus.pdf...
32 segments stockés avec succès
  Chargement de [Cours] Utilisation de Réseaux existants.pdf...
40 segments stockés avec succès
  Chargement de Apprentissage par transfert [Tensorflow].pdf...
62 segments stockés avec succès
  Chargement de Cnn _Univ-batan2.pdf...
41 segments stockés avec succès
  Chargement de Initiation au Deep Learning _ Joel kazadi.pdf...
170 segments stockés avec succès
  Chargement de CNN Aix Marseille Université.pdf...
29 segments stockés avec succès
  Chargement de [Cours] _Transformers.pdf...
59 segments stockés avec succès
  Chargement de deep learning avec pytorch univ_lyon2.pdf...
63 segments stockés avec succès
  Charg